# vnocr — huấn luyện recogniser tiếng Việt trên Kaggle (GPU free)

**Trước khi Run All, bật 2 thứ ở panel bên phải:**
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100).
2. **Settings → Internet → On** (để `git clone` + tải font).

Notebook tự lo: lấy code → font tiếng Việt → sinh ảnh synthetic → smoke test →
train trên GPU → lưu `recognizer.pt` vào Output để tải về.

> Corpus/ảnh mẫu chỉ đủ **chứng minh pipeline chạy**. Để có model đọc ảnh thật
> tốt: gắn thêm Kaggle Dataset corpus lớn (Wikipedia+báo) và tăng `N_SYNTH`,
> `EPOCHS` — xem `docs/TRAINING.md`.

## 1. Cấu hình

In [ ]:
# --- Sửa các giá trị này nếu cần ---
REPO_URL = 'https://github.com/dangquoc123/toanquoc-ocr.git'
# Để REPO_URL = '' nếu muốn dùng repo gắn dưới dạng Kaggle Dataset thay vì GitHub.
CORPUS_PATH = ''  # để trống -> dùng data/corpus/vi_sample.txt trong repo.
                  # Trỏ tới corpus lớn nếu bạn gắn dataset, vd:
                  # '/kaggle/input/vi-corpus/corpus.txt'

N_SYNTH = 8000    # số ảnh synthetic (tăng lên 100k+ cho model thật)
EPOCHS  = 12      # tăng lên 30+ cho model thật
BATCH   = 128
USE_GTC = True    # Guided Training of CTC (§3.1)

## 2. Lấy code vnocr
Ưu tiên `git clone`. Nếu `REPO_URL` để trống, notebook tìm repo trong
`/kaggle/input/*` (gắn cả thư mục repo làm Dataset).

In [ ]:
import os, sys, glob, shutil, subprocess
WORK = '/kaggle/working'
REPO = os.path.join(WORK, 'vnocr-repo')

if REPO_URL:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO], check=True)
else:
    # tìm pyproject.toml của vnocr trong các dataset đã gắn
    hits = glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
    src = None
    for h in hits:
        if 'vnocr' in open(h).read():
            src = os.path.dirname(h); break
    assert src, ('Không thấy repo. Điền REPO_URL hoặc gắn thư mục repo làm '
                 'Kaggle Dataset (Add Input).')
    if os.path.isdir(REPO):
        shutil.rmtree(REPO)
    shutil.copytree(src, REPO)

os.chdir(REPO)
sys.path.insert(0, REPO)
import vnocr
import torch
print('vnocr', vnocr.__version__, '| repo', REPO)
print('CUDA :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Bật GPU: Settings -> Accelerator -> GPU T4'

## 3. Font tiếng Việt (§6.2)
Dùng font hệ thống của Kaggle (DejaVu phủ đủ dấu). Tải thêm Noto nếu có mạng.

In [ ]:
sys.path.insert(0, os.path.join(REPO, 'scripts'))
from build_synth_data import find_fonts, verify_font

FONTS_DIR = os.path.join(WORK, 'fonts')
os.makedirs(FONTS_DIR, exist_ok=True)
# copy các font hệ thống phủ đủ dấu vào một thư mục
good = [f for f in find_fonts('/usr/share/fonts') if verify_font(f)]
for f in good:
    try: shutil.copy(f, FONTS_DIR)
    except Exception: pass

# (tuỳ chọn) tải Noto Sans để đa dạng font — bỏ qua nếu không có mạng
try:
    subprocess.run(['wget', '-q', '-O', os.path.join(FONTS_DIR, 'NotoSans.ttf'),
        'https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf'],
        check=True, timeout=60)
except Exception as e:
    print('bỏ qua Noto:', e)

usable = [f for f in find_fonts(FONTS_DIR) if verify_font(f)]
print(f'{len(usable)} font phủ đủ dấu tiếng Việt')
assert usable, 'Không có font hợp lệ'

## 4. Corpus + đo entropy (§8)

In [ ]:
CORPUS = CORPUS_PATH or os.path.join(REPO, 'data/corpus/vi_sample.txt')
print('corpus:', CORPUS)
!python scripts/measure_entropy.py "$CORPUS"

## 5. Smoke test — kiểm máy móc trước khi train
Overfit 1 batch ngẫu nhiên, kiểm shape/loss/decode. Phải in **SMOKE TEST PASSED**.

In [ ]:
!python scripts/smoke_train.py --steps 200
# muốn kiểm luôn nhánh teacher: thêm --gtc

## 6. Sinh ảnh synthetic (§6.2)

In [ ]:
SYNTH = os.path.join(WORK, 'synth')
rc = subprocess.run([sys.executable, 'scripts/build_synth_data.py',
    '--corpus', CORPUS, '--fonts', FONTS_DIR,
    '--out', SYNTH, '--n', str(N_SYNTH), '--mode', 'mixed']).returncode
assert rc == 0, 'build_synth_data lỗi'

## 7. Chia train / val (95 / 5)

In [ ]:
import random
labels = open(os.path.join(SYNTH, 'labels.txt'), encoding='utf-8').read().splitlines()
random.Random(0).shuffle(labels)
k = max(1, int(len(labels) * 0.95))
open(os.path.join(SYNTH, 'train.txt'), 'w', encoding='utf-8').write('\n'.join(labels[:k]))
open(os.path.join(SYNTH, 'val.txt'),   'w', encoding='utf-8').write('\n'.join(labels[k:]))
print('train', k, '| val', len(labels) - k)

## 8. Charset + language model cho hậu xử lý (§5)

In [ ]:
!python scripts/build_charset.py --out data/charset/syllables.txt
!python scripts/train_lm.py "$CORPUS" --order 3 --out {WORK}/vi.count.pkl

## 9. Huấn luyện recogniser trên GPU (§3, §7)
Mỗi epoch in CER/WER và **p_B/p_T** trên tập val. `recognizer.pt` lưu vào Output.

In [ ]:
CKPT = os.path.join(WORK, 'recognizer.pt')
cmd = [sys.executable, 'scripts/train_recognizer.py',
       '--train', f'{SYNTH}/train.txt', '--val', f'{SYNTH}/val.txt',
       '--epochs', str(EPOCHS), '--batch', str(BATCH),
       '--device', 'cuda', '--out', CKPT]
if USE_GTC:
    cmd.append('--gtc')
print(' '.join(cmd))
rc = subprocess.run(cmd).returncode
assert rc == 0, 'train_recognizer lỗi'

## 10. Thử đọc một ảnh (kiểm end-to-end)

In [ ]:
import glob
sample = sorted(glob.glob(os.path.join(SYNTH, 'images', '*.png')))[0]
rc = subprocess.run([sys.executable, 'scripts/infer.py', sample,
    '--ckpt', CKPT,
    '--syllables', 'data/charset/syllables.txt',
    '--lm', os.path.join(WORK, 'vi.count.pkl'),
    '--format', 'text']).returncode
from IPython.display import Image, display
display(Image(sample))
labels = dict(l.split('\t', 1) for l in
              open(os.path.join(SYNTH, 'labels.txt'), encoding='utf-8')
              .read().splitlines() if '\t' in l)
print('nhãn gốc:', labels.get('images/' + os.path.basename(sample)))

## 11. Tải kết quả về
Ở tab **Output** (hoặc `/kaggle/working`) tải:
- `recognizer.pt` — model đã train
- `vi.count.pkl` — language model

Rồi dùng lại local:
```bash
python3 scripts/infer.py page.jpg --ckpt recognizer.pt \
    --syllables data/charset/syllables.txt --lm vi.count.pkl --format json
```

**Để lên chất lượng thật:** tăng `N_SYNTH` (100k+), `EPOCHS` (30+), gắn corpus
lớn + dataset thật (VinText), và cân nhắc chạy nhiều phiên (Kaggle 30h/tuần).